Cronometraje deportivo - Multiesport
Empresa de cronometraje deportivos - Yomurycronometraje
-- Resultados | Cronofinisher
Resultados – JCJ Timing
Cronochip - Cronometraje de carreras
Pantalla Principal - CRONORFID
CronoTech Carreras
SOFTWARE Clasificaciones - CRONOMACH
MDG-Carreras - MdgSoft.com
DSport Sistema de Cronometratge

In [88]:
# -*- coding: utf-8 -*-
"""
Parser flexible de PDFs de classificacions de Cronofinisher.

Dissenyat per aguantar la variabilitat observada entre organitzadors:
  - Noms de columna diferents (Tiempo/Meta, Apellido/Apellidos, PCAT/"Pos. categ.")
  - 4 convencions de sexe diferents:
      1) paraula completa dins Categoria ("Masculino"/"Femenino", "Hombres"/"Mujeres")
      2) abreviatura dins Categoria ("MASC"/"FEM")
      3) lletra sola al final de Categoria ("ELITE M", "MASTER 40 M", "ELITE F")
      4) sexe pel NOM DEL FITXER/enllaç PDF, no per fila ("GENERAL MASCULINA.pdf")
  - Bloc d'estadístiques de capçalera opcional (INSCRITOS/META/RETIRADOS/
    DESCALIFICADOS/NO PRESENTADOS/CORTE) que, quan existeix, és més fiable
    que comptar files.
  - PDFs de classificació múltiples per esdeveniment (per modalitat de
    distància CORTA/LARGA, per tipus de participació INDIVIDUAL/PAREJAS/
    EBIKE, per categoria d'edat PITUFOS/PREBENJAMIN/BENJAMIN, o per nivell
    OPEN/AVANZADO) que cal tractar com a files separades.
  - Esdeveniments sense cap PDF de classificació completa utilitzable
    (només PODIUM/TROFEOS/rànquings de club).

Política conservadora: si no es pot determinar el sexe d'una fila amb cap
dels 4 mètodes, es compta a `sexe_desconegut` (mai s'endevina).
"""
import re
import unicodedata
from dataclasses import dataclass, field
from typing import Optional


# ---------------------------------------------------------------------------
# Normalització i alies de columnes
# ---------------------------------------------------------------------------

def normalize(s) -> str:
    """minúscules, sense accents, sense puntuació/espais -> comparació robusta."""
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


# canonical_name -> conjunt de variants normalitzades conegudes
COLUMN_ALIASES = {
    "pos_general": {"pos", "posgral", "posicion", "posicióngeneral", "posgeneral"},
    "pos_genero": {"pgen", "posgenero", "possexo", "possexo"},
    "pos_categoria": {"pcat", "poscateg", "poscategoria", "poscategoría"},
    "dorsal": {"dorsal", "dor"},
    "nombre": {"nombre"},
    "apellido": {"apellido", "apellidos"},
    "apellido_nombre_combinado": {"apellidosnombre"},
    "club": {"club"},
    "categoria": {"categoria", "cat"},
    "tiempo": {"tiempo", "meta"},
    "ritmo": {"ritmo", "vel"},
}
# invertim per lookup O(1)
_ALIAS_LOOKUP = {}
for canon, variants in COLUMN_ALIASES.items():
    for v in variants:
        _ALIAS_LOOKUP[v] = canon


def detect_columns(header_cells) -> dict:
    """header_cells: llista de strings (una fila de capçalera d'una taula pdfplumber).
    Retorna dict canonical_name -> índex de columna. Es queda la 1a coincidència
    (evita que columnes repetides d'un desglossament per disciplina -natació/
    ciclisme/carrera- sobreescriguin la columna general 'pos')."""
    mapping = {}
    for idx, cell in enumerate(header_cells or []):
        key = normalize(cell)
        canon = _ALIAS_LOOKUP.get(key)
        if canon and canon not in mapping:
            mapping[canon] = idx
    return mapping


# ---------------------------------------------------------------------------
# Cascada de detecció de sexe (4 nivells, del més al menys específic)
# ---------------------------------------------------------------------------

_GENDER_FULLWORD = [
    (re.compile(r"\bfemenin[oa]s?\b", re.IGNORECASE), "d"),
    (re.compile(r"\bmasculin[oa]s?\b", re.IGNORECASE), "h"),
    (re.compile(r"\bmujere?s?\b", re.IGNORECASE), "d"),
    (re.compile(r"\bhombres?\b", re.IGNORECASE), "h"),
]
_GENDER_ABBREV = [
    (re.compile(r"\bfem\b", re.IGNORECASE), "d"),
    (re.compile(r"\bmasc\b", re.IGNORECASE), "h"),
]
_GENDER_SINGLE_LETTER = [
    (re.compile(r"\bf$", re.IGNORECASE), "d"),
    (re.compile(r"\bm$", re.IGNORECASE), "h"),
]


def detect_gender_from_text(text: Optional[str]) -> Optional[str]:
    """Prova la cascada (paraula completa -> abreviatura -> lletra sola)
    sobre un sol text (normalment el valor de la cel·la 'Categoria').
    Retorna 'h', 'd' o None si no es reconeix cap patró."""
    if not text:
        return None
    text = text.strip()
    for pattern, g in _GENDER_FULLWORD:
        if pattern.search(text):
            return g
    for pattern, g in _GENDER_ABBREV:
        if pattern.search(text):
            return g
    for pattern, g in _GENDER_SINGLE_LETTER:
        if pattern.search(text):
            return g
    return None


def detect_gender_from_filename(filename_or_link_text: Optional[str]) -> Optional[str]:
    """4t nivell: sexe determinat pel nom del fitxer/enllaç PDF sencer
    (p.ex. 'CLASIFICACION GENERAL MASCULINA' -> 'h', 'OPEN MASC' -> 'h').
    Reutilitza la mateixa cascada de paraules completes/abreviatures
    (la lletra sola es desactiva aquí per evitar falsos positius amb
    paraules de nom de fitxer que acabin en m/f per atzar)."""
    if not filename_or_link_text:
        return None
    text = filename_or_link_text.strip()
    for pattern, g in _GENDER_FULLWORD:
        if pattern.search(text):
            return g
    for pattern, g in _GENDER_ABBREV:
        if pattern.search(text):
            return g
    return None


# ---------------------------------------------------------------------------
# Bloc d'estadístiques de capçalera (format "triatló Cronofinisher")
# ---------------------------------------------------------------------------

_STATS_PATTERNS = {
    "inscrits": re.compile(r"INSCRITOS[:\s]+(\d+)", re.IGNORECASE),
    "total_classificats": re.compile(r"\bMETA[:\s]+(\d+)", re.IGNORECASE),
    "DNF_total": re.compile(r"RETIRADOS[^\d]{0,15}(\d+)", re.IGNORECASE),
    "DSQ_total": re.compile(r"DESCALIFICADOS[^\d]{0,15}(\d+)", re.IGNORECASE),
    "DNS_total": re.compile(r"NO\s*PRESENTADOS[^\d]{0,15}(\d+)", re.IGNORECASE),
}


def extract_header_stats(full_text: str) -> dict:
    """Cerca el bloc 'INSCRITOS: N META: N RETIRADOS (DNF): N ...' si existeix.
    Retorna dict parcial o buit si no es troba cap patró (format sense capçalera
    d'estadístiques, com el trail o la cursa popular)."""
    stats = {}
    if not full_text:
        return stats
    for field_name, pattern in _STATS_PATTERNS.items():
        m = pattern.search(full_text)
        if m:
            stats[field_name] = int(m.group(1))
    return stats


# ---------------------------------------------------------------------------
# Filtratge de quins PDFs de la pestanya "Clasificaciones" cal parsejar
# ---------------------------------------------------------------------------
# IMPORTANT (revisat després de veure events reals addicionals): la primera
# versió exigia que el nom contingués "general". Això era fals: events com
# "Dual Battle Guadalquivir" anomenen els seus PDFs "OPEN MASC"/"AVANZADO
# FEM"/"OPEN MIXTO" (sense la paraula "general" enlloc), i amb l'exigència
# antiga cap PDF de l'esdeveniment passava el filtre -> tot l'esdeveniment
# es marcava (incorrectament) `sense_resultats`. Ara només s'exclouen els
# PDFs que sabem del cert que són subconjunts (trofeus/podi) o rànquings
# agregats de clubs -- la resta es dona per bo. Els PDFs que resultin no ser
# taules reals (diplomes, certificats...) ja es descarten més endavant, quan
# `_valida_parsing_pdf` (al scraper) detecta que no s'hi ha trobat cap taula.
#
# NOTA sobre "equipo": s'ha exclòs deliberadament de la llista negra. Un cas
# real (Maratón Ciudad de Jaén, PDF "GENERAL MARATON POR EQUIPOS" / fitxer
# "...tiempos-de-cada-relevista.pdf") demostra que "equipos" de vegades és
# una modalitat real de participació (relleus, un temps per persona), no un
# rànquing agregat de clubs -- excloure-la per defecte perdria dades bones.
# Però això vol dir que un ranquing agregat real de clubs/equips que faci
# servir aquesta paraula ja NO es filtrarà aquí: si compta files, comptarà
# equips com si fossin persones. És un risc conegut i acceptat; si es
# detecten xifres estranyament baixes en una modalitat amb "equipo" al nom,
# val la pena revisar-ne el PDF a mà.
#
# NOTA sobre "categoria(s)": AFEGIT després de comparar el contingut real de
# parelles de PDFs "CLASIFICACION GENERAL" + "CATEGORIAS" del mateix esdeve-
# niment (II CxM Rompealbarcas, Carrera Popular San Martín del Tesorillo,
# Duatlón Cros Jerez de los Caballeros, Desafío La Codosera). En tots els
# casos "CATEGORIAS" resulta ser un top-N-per-categoria -- EXACTAMENT les
# mateixes primeres persones que ja apareixen a "CLASIFICACION GENERAL" --
# és a dir, funciona igual que un "TROFEOS" amb un altre nom, no una llista
# addicional. Si es deixés passar, es comptaria com una modalitat pròpia i
# duplicaria/falsejaria dades que ja es compten bé des del fitxer "GENERAL"
# corresponent.
_EXCLUDED_PDF_KEYWORDS = ["trofeo", "podium", "podio", "club", "categoria"]


def is_general_classification_pdf(link_text: str) -> bool:
    """True si l'enllaç sembla una llista completa de classificats (i no un
    podi, un trofeu, un top-N per categoria, o un rànquing agregat de
    clubs)."""
    if not link_text or not link_text.strip():
        return False
    norm = normalize(link_text)
    return not any(kw in norm for kw in _EXCLUDED_PDF_KEYWORDS)


# ---------------------------------------------------------------------------
# Recompte per files (fallback quan no hi ha bloc d'estadístiques,
# o verificació creuada quan sí que n'hi ha)
# ---------------------------------------------------------------------------

@dataclass
class ClassificationCounts:
    total: int = 0
    h: int = 0
    d: int = 0
    sexe_desconegut: int = 0


def _row_is_empty(row) -> bool:
    return not row or all(c is None or str(c).strip() == "" for c in row)


def count_from_rows(rows, col_map: dict, file_gender: Optional[str] = None) -> ClassificationCounts:
    """rows: llista de files (cadascuna, llista de cel·les) SENSE la capçalera.
    file_gender: si el PDF sencer és d'un sol sexe (patró 4, per nom de fitxer),
    es passa aquí i s'aplica a totes les files sense mirar la columna Categoria."""
    counts = ClassificationCounts()
    cat_idx = col_map.get("categoria")
    for row in rows:
        if _row_is_empty(row):
            continue
        counts.total += 1
        if file_gender in ("h", "d"):
            g = file_gender
        elif cat_idx is not None and cat_idx < len(row):
            g = detect_gender_from_text(row[cat_idx])
        else:
            g = None
        if g == "h":
            counts.h += 1
        elif g == "d":
            counts.d += 1
        else:
            counts.sexe_desconegut += 1
    return counts

In [89]:
# -*- coding: utf-8 -*-
"""
Tests amb taules "de mentida" que reprodueixen casos reals vistos al
navegador a cronofinisher.com (sense dependre de xarxa/sandbox).
"""
import unittest
# (detect_columns, detect_gender_from_text, etc. ja estan definides a la
#  cel·la 1 d'aquest mateix notebook -- cal haver-la executat abans;
#  no cal ni es pot importar-les d'un fitxer extern)


class TestDeteccioColumnes(unittest.TestCase):

    def test_cas1_trail_sierra_magina(self):
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Apellido",
                   "Club", "Categoria", "Tiempo", "Ritmo"]
        cols = detect_columns(header)
        self.assertEqual(cols["pos_general"], 0)
        self.assertEqual(cols["pos_genero"], 1)
        self.assertEqual(cols["pos_categoria"], 2)
        self.assertEqual(cols["club"], 6)
        self.assertEqual(cols["categoria"], 7)
        self.assertEqual(cols["tiempo"], 8)
        self.assertEqual(cols["ritmo"], 9)

    def test_cas2_popular_malpartida(self):
        header = ["Pos", "Dorsal", "Nombre", "Apellidos", "Categoría",
                   "Pos. categ.", "Meta"]
        cols = detect_columns(header)
        self.assertEqual(cols["pos_general"], 0)
        self.assertEqual(cols["apellido"], 3)
        self.assertEqual(cols["categoria"], 4)
        self.assertEqual(cols["pos_categoria"], 5)
        self.assertEqual(cols["tiempo"], 6)  # "Meta" -> alias de tiempo
        self.assertNotIn("pos_genero", cols)  # aquesta cursa no en té
        self.assertNotIn("club", cols)        # aquesta cursa no en té

    def test_cas3_ciclismo_conquista_magina(self):
        # mateixa estructura que cas 1
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Apellido",
                   "Club", "Categoria", "Tiempo", "Ritmo"]
        cols = detect_columns(header)
        self.assertEqual(len(cols), 10)

    def test_columna_desconeguda_no_trenca_res(self):
        header = ["POS", "ALGUNA_COSA_NOVA", "Dorsal"]
        cols = detect_columns(header)
        self.assertEqual(cols["pos_general"], 0)
        self.assertEqual(cols["dorsal"], 2)
        self.assertEqual(len(cols), 2)

    def test_cas6_dual_battle_sense_apellido_ni_club(self):
        # PDF real ("OPEN MASC", Dual Battle Guadalquivir): capçalera més
        # curta, sense Apellido ni Club (la columna "Nombre" hi porta el
        # nom de l'equip sencer).
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Categoria", "Tiempo"]
        cols = detect_columns(header)
        self.assertEqual(cols["pos_general"], 0)
        self.assertEqual(cols["dorsal"], 3)
        self.assertEqual(cols["nombre"], 4)
        self.assertEqual(cols["categoria"], 5)
        self.assertEqual(cols["tiempo"], 6)
        self.assertNotIn("apellido", cols)
        self.assertNotIn("club", cols)


class TestDeteccioSexe(unittest.TestCase):

    # --- Nivell 1: paraula completa (cas 2, Malpartida) ---
    def test_paraula_completa(self):
        self.assertEqual(detect_gender_from_text("SENIOR Masculino"), "h")
        self.assertEqual(detect_gender_from_text("VETERANO Femenino"), "d")

    # --- Nivell 2: abreviatura (cas 1, Sierra Mágina) ---
    def test_abreviatura(self):
        self.assertEqual(detect_gender_from_text("ABS MASC"), "h")
        self.assertEqual(detect_gender_from_text("JUVENIL FEM"), "d")
        self.assertEqual(detect_gender_from_text("DIVERSIDAD FUNC MASC"), "h")

    # --- Nivell 3: lletra sola (cas 3, Conquista Mágina) ---
    def test_lletra_sola(self):
        self.assertEqual(detect_gender_from_text("ELITE M"), "h")
        self.assertEqual(detect_gender_from_text("MASTER 40 M"), "h")
        self.assertEqual(detect_gender_from_text("ELITE F"), "d")
        self.assertEqual(detect_gender_from_text("SUB-23 M"), "h")

    # --- Nivell 4: nom de fitxer (cas 4, Trihercules; cas 6, Dual Battle) ---
    def test_per_nom_fitxer(self):
        self.assertEqual(
            detect_gender_from_filename("CLASIFICACION GENERAL MASCULINA"), "h")
        self.assertEqual(
            detect_gender_from_filename("CLASIFICACION GENERAL FEMENINA"), "d")
        self.assertEqual(detect_gender_from_filename("OPEN MASC"), "h")
        self.assertEqual(detect_gender_from_filename("AVANZADO FEM"), "d")
        self.assertIsNone(detect_gender_from_filename("OPEN MIXTO"))

    # --- Casos que han de quedar com a None (mai endevinar) ---
    def test_no_reconegut_retorna_none(self):
        self.assertIsNone(detect_gender_from_text(""))
        self.assertIsNone(detect_gender_from_text(None))
        self.assertIsNone(detect_gender_from_text("MIXTO"))
        self.assertIsNone(detect_gender_from_text("EQUIPOS"))

    def test_categoria_curta_no_dona_fals_positiu_lletra_sola(self):
        # "M" sol -> coincideix igualment (\bm$ actua igual); ho acceptem
        # perquè en la pràctica Cronofinisher sempre l'escriu així.
        self.assertEqual(detect_gender_from_text("M"), "h")


class TestBlocEstadistiquesCapcalera(unittest.TestCase):

    def test_format_triatlo_trihercules(self):
        text = (
            "VIII TRIHERCULES CADIZ 2024\n"
            "INSCRITOS: 234   META: 194\n"
            "RETIRADOS (DNF): 2   DESCALIFICADOS (DSQ): 0   "
            "NO PRESENTADOS (DNS): 38   CORTE (NC): 0\n"
            "CLASIFICACION GENERAL MASCULINA\n"
        )
        stats = extract_header_stats(text)
        self.assertEqual(stats["inscrits"], 234)
        self.assertEqual(stats["total_classificats"], 194)
        self.assertEqual(stats["DNF_total"], 2)
        self.assertEqual(stats["DSQ_total"], 0)
        self.assertEqual(stats["DNS_total"], 38)

    def test_sense_bloc_capcalera_retorna_buit(self):
        text = "CLASIFICACIÓN GENERAL\nCXM CORTA\nPOS PGEN PCAT ..."
        stats = extract_header_stats(text)
        self.assertEqual(stats, {})


class TestFiltratgePdfsClassificacio(unittest.TestCase):

    def test_accepta_amb_paraula_general(self):
        self.assertTrue(is_general_classification_pdf("GENERAL CORTA"))
        self.assertTrue(is_general_classification_pdf("CLASIFICACIÓN GENERAL 5K"))
        self.assertTrue(is_general_classification_pdf("GENERAL INDIVIDUAL"))
        self.assertTrue(is_general_classification_pdf("GENERAL PAREJAS"))
        self.assertTrue(is_general_classification_pdf("CLASIFICACION GENERAL MASCULINA"))
        self.assertTrue(is_general_classification_pdf("GENERAL PITUFOS"))

    def test_accepta_sense_paraula_general(self):
        # CAS REAL confirmat (Dual Battle Guadalquivir): cap d'aquests PDFs
        # conté la paraula "general" i tots són classificacions completes.
        # La versió anterior del filtre els rebutjava TOTS -> l'esdeveniment
        # sencer quedava marcat sense_resultats per error.
        self.assertTrue(is_general_classification_pdf("OPEN MASC"))
        self.assertTrue(is_general_classification_pdf("OPEN FEM"))
        self.assertTrue(is_general_classification_pdf("OPEN MIXTO"))
        self.assertTrue(is_general_classification_pdf("AVANZADO MASC"))

    def test_accepta_equipos_com_a_modalitat_real(self):
        # CAS REAL confirmat (Maraton Ciudad de Jaen): "GENERAL MARATON POR
        # EQUIPOS" és una modalitat de relleus amb un temps per persona, no
        # un rànquing agregat de clubs -- ja no s'exclou per la paraula
        # "equipo" (vegeu la nota al mòdul sobre el risc conegut que això
        # comporta per a rànquings agregats reals que facin servir la
        # mateixa paraula).
        self.assertTrue(is_general_classification_pdf("GENERAL MARATON POR EQUIPOS"))

    def test_rebutja_trofeus_podium_i_clubs(self):
        self.assertFalse(is_general_classification_pdf("TROFEOS CORTA"))
        self.assertFalse(is_general_classification_pdf("PODIUM"))
        self.assertFalse(is_general_classification_pdf("TROFEO ADULTO"))
        self.assertFalse(is_general_classification_pdf("CLASIFICACION CLUBES MASCULINOS"))

    def test_rebutja_categorias_com_a_top_n_duplicat(self):
        # CAS REAL confirmat comparant el contingut: "LA ALBARQUILLA -
        # CATEGORIAS" (1 pàgina) conté EXACTAMENT les 3 mateixes primeres
        # persones que ja surten a "LA ALBARQUILLA - CLASIFICACION GENERAL"
        # (6 pàgines) -- és un top-3-per-categoria, no una llista addicional.
        # Deixar-lo passar duplicaria/falsejaria el recompte.
        self.assertFalse(is_general_classification_pdf("LA ALBARQUILLA - CATEGORIAS"))
        self.assertFalse(is_general_classification_pdf("ABSOLUTA CATEGORÍAS"))
        self.assertFalse(is_general_classification_pdf("CATEGORIAS"))

    def test_rebutja_text_buit(self):
        self.assertFalse(is_general_classification_pdf(""))
        self.assertFalse(is_general_classification_pdf(None))

    def test_esdeveniment_sense_pdf_valid(self):
        # cas real: IV Triatlón Playa de Peloche -> només PODIUM
        links = ["PODIUM"]
        valids = [l for l in links if is_general_classification_pdf(l)]
        self.assertEqual(valids, [])  # -> ha de marcar-se sense_resultats


class TestRecompteFiles(unittest.TestCase):

    def test_cas1_trail_sierra_magina_amb_abreviatura(self):
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Apellido",
                   "Club", "Categoria", "Tiempo", "Ritmo"]
        rows = [
            ["1", "1", "1", "128", "Diego", "Robles Morales", "CLUB X", "ABS MASC", "01:03:42", "5'18/Km"],
            ["7", "7", "1", "109", "Francisco Javier", "Deogracia Jimenez", "IND.", "DIVERSIDAD FUNC MASC", "01:14:25", "6'12/Km"],
            ["18", "1", "1", "174", "Claudia", "Rodríguez Castillo", "TRAILRUNNERS", "JUVENIL FEM", "01:25:54", "7'09/Km"],
            ["20", "2", "1", "147", "Marta", "González De Miguel Guerrero", "ANTORCHA", "ABS FEM", "01:27:51", "7'19/Km"],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 4)
        self.assertEqual(counts.h, 2)
        self.assertEqual(counts.d, 2)
        self.assertEqual(counts.sexe_desconegut, 0)

    def test_cas2_popular_malpartida_paraula_completa(self):
        header = ["Pos", "Dorsal", "Nombre", "Apellidos", "Categoría",
                   "Pos. categ.", "Meta"]
        rows = [
            ["1", "3", "Enrique", "Ruiz Paredes", "SENIOR Masculino", "1", "00:18:18"],
            ["4", "8", "Ibai", "Etxabe De Domingo", "VETERANO Masculino", "1", "00:20:12"],
            ["8", "134", "Nerea", "Rodriguez Corchero", "SENIOR Femenino", "1", "00:20:36"],
            ["24", "59", "Maria De La Luz", "Hisado Padilla", "VETERANO Femenino", "1", "00:26:36"],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 4)
        self.assertEqual(counts.h, 2)
        self.assertEqual(counts.d, 2)

    def test_cas3_ciclismo_lletra_sola(self):
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Apellido",
                   "Club", "Categoria", "Tiempo", "Ritmo"]
        rows = [
            ["1", "1", "1", "1", "Francisco Javier", "Poza Ruiz", "KANINA BIKES", "ELITE M", "01:52:46", "21.28 Km/h"],
            ["2", "2", "1", "2", "Jose Manuel", "Hidalgo", "INDEPENDIENTE", "MASTER 30 M", "01:58:55", "20.18 Km/h"],
            ["21", "2", "1", "14", "Alexander", "Artymovych Marchak", "INDEPENDIENTE", "SUB-23 M", "02:31:43", "15.82 Km/h"],
            ["29", "1", "1", "151", "Mariló", "Herrera Poyatos", "KANINA BIKES", "ELITE F", "02:39:44", "15.02 Km/h"],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 4)
        self.assertEqual(counts.h, 3)
        self.assertEqual(counts.d, 1)
        self.assertEqual(counts.sexe_desconegut, 0)

    def test_cas4_sense_file_gender_cauria_a_desconegut(self):
        # DOCUMENTA UN CAS LÍMIT IMPORTANT: codis sense separador ("23M",
        # "SNM", "V1M") NO els detecta la cascada de "lletra sola" perquè
        # no hi ha cap límit de paraula entre el dígit/lletra i la M/F final.
        # Per això, per aquest format, el scraper HA de detectar el sexe
        # pel nom del fitxer (nivell 4) i passar-lo com file_gender -- no
        # es pot confiar només en la columna Categoria.
        header = ["POS", "Dor", "Nombre", "Club", "Cat.", "P.Cat.", "Tiempo"]
        rows = [
            ["1", "6", "Jesus Vela Vela", "PEÑOTA DENTAL", "23M", "1", "00:56:24"],
        ]
        col_map = detect_columns(header)
        counts_sense_override = count_from_rows(rows, col_map)  # sense file_gender
        self.assertEqual(counts_sense_override.h, 0)
        self.assertEqual(counts_sense_override.sexe_desconegut, 1)  # es perd!

    def test_cas4_triatlo_fitxer_separat_per_sexe(self):
        # En aquest format el sexe NO surt a la columna Categoria (23M, SNM,
        # V1M...) sinó que ve donat pel nom del fitxer -> s'ha de passar
        # file_gender explícitament, no confiar en la columna.
        header = ["POS", "Dor", "Nombre", "Club", "Cat.", "P.Cat.", "Tiempo"]
        rows = [
            ["1", "6", "Jesus Vela Vela", "PEÑOTA DENTAL", "23M", "1", "00:56:24"],
            ["2", "19", "Alfonso Bastos Garcia", "CLUB TRIATLON", "SNM", "1", "00:59:14"],
        ]
        col_map = detect_columns(header)
        file_gender = detect_gender_from_filename(
            "29092423324873202024-09-28-cadiz-general-masculina.pdf")
        self.assertEqual(file_gender, "h")
        counts = count_from_rows(rows, col_map, file_gender=file_gender)
        self.assertEqual(counts.total, 2)
        self.assertEqual(counts.h, 2)
        self.assertEqual(counts.d, 0)

    def test_cas5_parejas_cada_fila_es_una_persona_no_una_parella(self):
        # CONFIRMAT contra el PDF real (general-parejas.pdf, event 694):
        # el dorsal porta sufix A/B per identificar cada membre, però cada
        # persona té la seva pròpia fila -> NO cal duplicar ni dividir res,
        # el recompte de files ja és el recompte de persones.
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Apellido",
                   "Club", "Categoria", "Tiempo", "Ritmo"]
        rows = [
            ["1", "1", "1", "232A", "Miguel", "Jiménez Torres", "SABIKA", "PAREJAS MASTER 40 M", "02:07:46", "18.78 Km/h"],
            ["2", "2", "2", "232B", "Adolfo", "García Quesada", "SABIKA", "PAREJAS MASTER 40 M", "02:07:46", "18.78 Km/h"],
            ["3", "3", "1", "210A", "Juan Antonio", "Castillo Ramirez", "BIOCYCLIST", "PAREJAS ELITE M", "02:11:19", "18.28 Km/h"],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 3)  # 3 persones (no 1.5 parelles!)
        self.assertEqual(counts.h, 3)

    def test_cas6_dual_battle_nom_equip_com_a_nombre(self):
        # PDF real ("open-masc.pdf", Dual Battle Guadalquivir): sense
        # Apellido ni Club, i "Nombre" porta el nom de l'equip sencer en
        # comptes d'un nom de persona -- no afecta el recompte, que només
        # mira Categoria/file_gender, mai la columna Nombre.
        header = ["POS", "PGEN", "PCAT", "Dorsal", "Nombre", "Categoria", "Tiempo"]
        rows = [
            ["1", "1", "1", "37", "Centro Híbrido", "OPEN MASC", "00:16:50"],
            ["2", "2", "2", "27", "Hidalgocorencia", "OPEN MASC", "00:17:05"],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 2)
        self.assertEqual(counts.h, 2)

    def test_categoria_no_reconeguda_va_a_desconegut(self):
        header = ["POS", "Dorsal", "Nombre", "Categoria"]
        rows = [
            ["1", "5", "X", "MIXTO"],       # no reconegut -> desconegut
            ["2", "6", "Y", "ABS MASC"],    # reconegut -> h
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 2)
        self.assertEqual(counts.sexe_desconegut, 1)
        self.assertEqual(counts.h, 1)

    def test_files_buides_no_compten(self):
        header = ["POS", "Dorsal", "Nombre", "Categoria"]
        rows = [
            ["1", "5", "X", "ABS MASC"],
            ["", "", "", ""],       # fila buida (peu de pàgina, etc.)
            [None, None, None, None],
        ]
        col_map = detect_columns(header)
        counts = count_from_rows(rows, col_map)
        self.assertEqual(counts.total, 1)


# exit=False perquè no intenti tancar el kernel de Jupyter en acabar
unittest.main(argv=[''], verbosity=2, exit=False)

test_format_triatlo_trihercules (__main__.TestBlocEstadistiquesCapcalera.test_format_triatlo_trihercules) ... ok
test_sense_bloc_capcalera_retorna_buit (__main__.TestBlocEstadistiquesCapcalera.test_sense_bloc_capcalera_retorna_buit) ... ok
test_cas1_trail_sierra_magina (__main__.TestDeteccioColumnes.test_cas1_trail_sierra_magina) ... ok
test_cas2_popular_malpartida (__main__.TestDeteccioColumnes.test_cas2_popular_malpartida) ... ok
test_cas3_ciclismo_conquista_magina (__main__.TestDeteccioColumnes.test_cas3_ciclismo_conquista_magina) ... ok
test_cas6_dual_battle_sense_apellido_ni_club (__main__.TestDeteccioColumnes.test_cas6_dual_battle_sense_apellido_ni_club) ... ok
test_columna_desconeguda_no_trenca_res (__main__.TestDeteccioColumnes.test_columna_desconeguda_no_trenca_res) ... ok
test_abreviatura (__main__.TestDeteccioSexe.test_abreviatura) ... ok
test_categoria_curta_no_dona_fals_positiu_lletra_sola (__main__.TestDeteccioSexe.test_categoria_curta_no_dona_fals_positiu_lletra_sola) ..

In [90]:
# -*- coding: utf-8 -*-
"""
Scraper de cronofinisher.com -- llistat + fitxa d'esdeveniment + descàrrega
i parsing dels PDFs de classificacions (fa servir les funcions de la
cel·la anterior: normalize, detect_columns, detect_gender_from_filename,
extract_header_stats, is_general_classification_pdf, count_from_rows).
"""
import os
import re
import csv
import io
import json
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import requests
from bs4 import BeautifulSoup
import pdfplumber

# ============================================================
# CONFIGURACIÓ
# ============================================================

BASE_URL = "https://www.cronofinisher.com"
LISTADO_URL = BASE_URL + "/resultados/"

CSV_PATH = Path("../../data/raw/cronofinisher/DF_CRONOFINISHER_SUCIO.csv")
CHECKPOINT_PATH = Path("../../data/raw/cronofinisher/cronofinisher_checkpoint.json")

RESUME = True
SAVE_EVERY_N_RACES = 20
REQUEST_DELAY_SECONDS = 1.0  # marge de cortesia entre peticions
MAX_PAGINES_LLISTAT = None   # None = totes les 37; posar un número baix per proves
MAX_REINTENTS_TRANSITORIS = 3

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

COLUMNES_IMPRESCINDIBLES = [
    "event_id", "nom_cursa", "data", "lloc", "modalitat_nom", "modalitat_codi",
    "estat_esdeveniment", "error_detall", "total_classificats",
    "classificat_h", "classificat_d", "classificat_total",
]
# Cronofinisher només dona el desglossament DNF/DSQ/DNS TOTAL (no per sexe)
# i només en alguns esdeveniments (format triatló amb bloc d'estadístiques
# de capçalera) -> no té sentit afegir columnes _h/_d que sempre quedarien
# buides per aquesta font.
COLUMNES_EXTRA = [
    "inscrits", "DNF_total", "DSQ_total", "DNS_total", "esport",
    "classificat_sexe_desconegut",
]
TOTES_LES_COLUMNES = COLUMNES_IMPRESCINDIBLES + COLUMNES_EXTRA


# ============================================================
# SESSIÓ HTTP -- cal una petició "d'escalfament" a la home/llistat
# abans de poder demanar fitxes d'esdeveniment (el lloc sembla
# resoldre rutes segons la sessió/cookies, no només per URL).
# ============================================================

def crea_sessio() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    s.get(BASE_URL + "/", timeout=30)  # estableix cookies de sessió
    return s


class ErrorTransitori(Exception):
    """Xarxa, 5xx -- NO es marca com fet, es reintenta en la propera execució."""


class ErrorPermanent(Exception):
    """404 o pàgina d'error real -- es desa una fila placeholder i mai es reintenta."""


def get_amb_reintents(session: requests.Session, url: str) -> requests.Response:
    """GET amb reintents pels errors transitoris (xarxa, 5xx).
    Un 404 es considera permanent i es propaga com a ErrorPermanent de seguida."""
    darrer_error = None
    for intent in range(1, MAX_REINTENTS_TRANSITORIS + 1):
        try:
            resp = session.get(url, timeout=30)
        except requests.RequestException as e:
            darrer_error = e
            time.sleep(REQUEST_DELAY_SECONDS * intent)
            continue
        if resp.status_code == 404:
            raise ErrorPermanent(f"404 a {url}")
        if resp.status_code >= 500:
            darrer_error = ErrorTransitori(f"{resp.status_code} a {url}")
            time.sleep(REQUEST_DELAY_SECONDS * intent)
            continue
        resp.raise_for_status()
        # Cronofinisher (PHP antic) no declara sempre el charset a la
        # capçalera Content-Type -> requests cau per defecte a ISO-8859-1
        # per a respostes text/html, cosa que trenca tots els accents i
        # cometes ("CÁDIZ" -> "CÃDIZ", """ -> "â€œ"). El contingut real
        # és UTF-8 (confirmat visualment al navegador), així que ho forcem
        # aquí, al punt únic per on passen totes les peticions HTML.
        # (No afecta les descàrregues de PDF: es llegeixen amb .content
        # en bytes, mai amb .text, així que aquest canvi els és indiferent.)
        resp.encoding = "utf-8"
        return resp
    raise ErrorTransitori(str(darrer_error) if darrer_error else f"error desconegut a {url}")


# ============================================================
# LLISTAT PAGINAT (/resultados/, /resultados/pg=N/)
# ============================================================

@dataclass
class EntradaLlistat:
    url: str
    nom_cursa: str
    data_text: str  # "09 AGO 2026" tal com surt al llistat


def parseja_pagina_llistat(html: str) -> list:
    """Extreu les entrades d'una pàgina del llistat /resultados/.
    Estructura confirmada al navegador: cada cursa és un
    div.container_padre_actividad amb .dia_evento/.mes_evento/.ano_evento
    i un <a> que embolcalla .titulo_evento_ficha."""
    soup = BeautifulSoup(html, "html.parser")
    entrades = []
    for card in soup.select(".container_padre_actividad"):
        titol_span = card.select_one(".titulo_evento_ficha")
        if not titol_span:
            continue
        link = titol_span.find_parent("a")
        if link is None:
            link = card.select_one("a[href*='/evento/']")
        if link is None or not link.get("href"):
            continue
        dia = card.select_one(".dia_evento")
        mes = card.select_one(".mes_evento")
        any_ = card.select_one(".ano_evento")
        data_text = " ".join(
            x.get_text(strip=True) for x in (dia, mes, any_) if x is not None
        )
        entrades.append(EntradaLlistat(
            url=link["href"].strip(),
            nom_cursa=titol_span.get_text(strip=True),
            data_text=data_text,
        ))
    return entrades


def _extreu_taules_pagina(pagina):
    """..."""
    return pagina.extract_tables(table_settings={
        "vertical_strategy": "text",
        "horizontal_strategy": "text",
    })

def normalitza_data_llistat(data_text: str) -> Optional[str]:
    """'09 AGO 2026' -> '2026-08-09'. Retorna None si no es pot parsejar
    (política conservadora, mai inventar una data)."""
    parts = data_text.split()
    if len(parts) != 3:
        return None
    dia, mes, any_ = parts
    mes_num = _MESOS.get(mes.upper())
    if not mes_num or not dia.isdigit() or not any_.isdigit():
        return None
    return f"{any_}-{mes_num}-{int(dia):02d}"


def crawl_llistat(session: requests.Session, max_pagines: Optional[int] = None):
    """Generador que dona (EntradaLlistat) per a cada cursa de tot el
    llistat, paginant automàticament. La primera pàgina determina el
    nombre total de pàgines."""
    resp = get_amb_reintents(session, LISTADO_URL)
    total_pagines = extreu_total_pagines(resp.text)
    if max_pagines:
        total_pagines = min(total_pagines, max_pagines)

    for entrada in parseja_pagina_llistat(resp.text):
        yield entrada
    time.sleep(REQUEST_DELAY_SECONDS)

    for pagina in range(2, total_pagines + 1):
        url = f"{BASE_URL}/resultados/pg={pagina}/"
        resp = get_amb_reintents(session, url)
        for entrada in parseja_pagina_llistat(resp.text):
            yield entrada
        time.sleep(REQUEST_DELAY_SECONDS)

In [91]:
# ============================================================
# FITXA D'ESDEVENIMENT: metadades + enllaços PDF + event_id
# ============================================================

@dataclass
class FitxaEsdeveniment:
    nom_cursa: str
    data: Optional[str]        # ISO YYYY-MM-DD, None si no es pot parsejar
    lloc: Optional[str]
    esport: Optional[str]      # "Tipo evento:" (Carrera, Trail, Triatlón...)
    event_id: Optional[str]    # numèric, tret de /biblioteca_eventos/<id>/
    pdf_links: list            # llista de (text_enllaç, url_pdf)


def _text_despres_de_strong(soup: BeautifulSoup, etiqueta: str) -> Optional[str]:
    """Cronofinisher fa servir <p><strong>Fecha:</strong> 03-05-2026</p> --
    agafem el text que ve immediatament després de l'etiqueta <strong>."""
    strong = soup.find("strong", string=lambda s: s and s.strip() == etiqueta)
    if strong is None or strong.next_sibling is None:
        return None
    text = strong.next_sibling.strip()
    return text or None


def normalitza_data_fitxa(data_text: Optional[str]) -> Optional[str]:
    """'03-05-2026' (DD-MM-YYYY, tal com surt a la fitxa) -> '2026-05-03'."""
    if not data_text:
        return None
    m = re.match(r"^(\d{2})-(\d{2})-(\d{4})$", data_text.strip())
    if not m:
        return None
    dia, mes, any_ = m.groups()
    return f"{any_}-{mes}-{dia}"


def parseja_fitxa_esdeveniment(html: str) -> FitxaEsdeveniment:
    soup = BeautifulSoup(html, "html.parser")

    h1 = soup.find("h1")
    nom_cursa = h1.get_text(strip=True) if h1 else ""

    data = normalitza_data_fitxa(_text_despres_de_strong(soup, "Fecha:"))
    lloc = _text_despres_de_strong(soup, "Lugar:")
    esport = _text_despres_de_strong(soup, "Tipo evento:")

    div_clas = soup.select_one("#info_clasificaciones")
    pdf_links = []
    event_id = None
    if div_clas is not None:
        for a in div_clas.select('a[href$=".pdf"]'):
            href = a.get("href", "").strip()
            if not href:
                continue
            pdf_links.append((a.get_text(strip=True), href))
            if event_id is None:
                m = re.search(r"biblioteca_eventos/(\d+)/", href)
                if m:
                    event_id = m.group(1)

    return FitxaEsdeveniment(
        nom_cursa=nom_cursa, data=data, lloc=lloc, esport=esport,
        event_id=event_id, pdf_links=pdf_links,
    )


# ============================================================
# AGRUPACIÓ DE PDFS "GENERAL" PER MODALITAT/SEXE
# ============================================================
# Patrons observats a les 8 curses investigades:
#   - "GENERAL CORTA" / "GENERAL LARGA"          -> 1 modalitat, 1 fitxer,
#                                                    sexe per fila (Categoria)
#   - "CLASIFICACIÓN GENERAL 5K"                  -> idem
#   - "GENERAL INDIVIDUAL"/"PAREJAS"/"EBIKE"      -> 3 modalitats, 1 fitxer
#                                                    cadascuna, sexe per fila
#   - "CLASIFICACION GENERAL MASCULINA"/"FEMENINA" -> 1 sola modalitat,
#                                                    partida en 2 fitxers
#                                                    (un per sexe) -> combinar

_PARAULES_SEXE = re.compile(
    r"\b(masculin[oa]s?|femenin[oa]s?|masc|fem|hombres?|mujere?s?)\b",
    re.IGNORECASE,
)
_PARAULES_SOROLL = re.compile(
    r"\b(clasificacion(es)?|clasificaci[oó]n|general)\b", re.IGNORECASE
)


def treu_modalitat_i_sexe_del_nom(link_text: str):
    """'CLASIFICACION GENERAL MASCULINA' -> ('', 'h')
       'GENERAL CORTA' -> ('CORTA', None)
       'GENERAL INDIVIDUAL' -> ('INDIVIDUAL', None)"""
    file_gender = detect_gender_from_filename(link_text)
    text = _PARAULES_SOROLL.sub(" ", link_text)
    text = _PARAULES_SEXE.sub(" ", text)
    modalitat = re.sub(r"\s+", " ", text).strip()
    return modalitat, file_gender


@dataclass
class GrupModalitat:
    modalitat_nom: str
    sense_genere: Optional[tuple] = None   # (text, url)
    masculina: Optional[tuple] = None      # (text, url)
    femenina: Optional[tuple] = None       # (text, url)


def agrupa_pdfs_per_modalitat(pdf_links) -> list:
    """pdf_links: llista de (text_enllaç, url) tal com surt de
    parseja_fitxa_esdeveniment(). Descarta TROFEOS/PODIUM/CLUBES/EQUIPOS
    (is_general_classification_pdf) i agrupa la resta per modalitat."""
    grups = {}
    for text, url in pdf_links:
        if not is_general_classification_pdf(text):
            continue
        modalitat, gender = treu_modalitat_i_sexe_del_nom(text)
        key = normalize(modalitat)
        grup = grups.setdefault(key, GrupModalitat(modalitat_nom=modalitat or "GENERAL"))
        if gender == "h":
            grup.masculina = (text, url)
        elif gender == "d":
            grup.femenina = (text, url)
        else:
            grup.sense_genere = (text, url)
    return list(grups.values())


# ============================================================
# DESCÀRREGA + PARSING D'UN PDF (integra cronofinisher_pdf_parser)
# ============================================================

def descarrega_i_parseja_pdf(session: requests.Session, url: str):
    """Retorna (llista de (header_cells, files) -- una parella per cada
    taula/pàgina trobada, cadascuna amb LA SEVA PRÒPIA capçalera --, text_complet).
    Llança ErrorTransitori/ErrorPermanent via get_amb_reintents.

    IMPORTANT (canvi respecte a la versió anterior): abans es feia servir
    la capçalera de la 1a taula trobada per interpretar TOTES les files de
    TOTES les pàgines. Però l'estratègia "text" de pdfplumber calcula les
    posicions de columna PÀGINA A PÀGINA, i poden no coincidir exactament
    d'una pàgina a la següent (confirmat: a Huesa, la pàgina 1 llegeix
    "Categoría" com una sola cel·la, la pàgina 2 la parteix en "Categ"/
    "oría"). Fer servir una capçalera global per a totes les pàgines feia
    que, als esdeveniments amb moltes pàgines, la columna "categoria" es
    llegís descol·locada a partir de la 2a pàgina -> gairebé tothom queia a
    `sexe_desconegut`. Ara cada taula es queda amb la seva pròpia capçalera,
    i qui la fa servir (`processa_modalitat`) calcula `detect_columns` per
    cadascuna per separat."""
    resp = get_amb_reintents(session, url)
    taules_amb_capcalera = []
    text_parts = []
    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        for pagina in pdf.pages:
            text_parts.append(pagina.extract_text() or "")
            for taula in _extreu_taules_pagina(pagina):
                if not taula:
                    continue
                taules_amb_capcalera.append((taula[0], taula[1:]))
    return taules_amb_capcalera, "\n".join(text_parts)

In [92]:
# ============================================================
# RESULTAT D'UNA MODALITAT (una fila del curses.csv)
# ============================================================

@dataclass
class ResultatModalitat:
    modalitat_nom: str
    modalitat_codi: str
    estat_esdeveniment: str   # ok / sense_resultats / llista_error / error
    error_detall: str = ""
    total_classificats: Optional[int] = None
    classificat_h: Optional[int] = None
    classificat_d: Optional[int] = None
    classificat_total: Optional[int] = None
    classificat_sexe_desconegut: Optional[int] = None
    inscrits: Optional[int] = None
    DNF_total: Optional[int] = None
    DSQ_total: Optional[int] = None
    DNS_total: Optional[int] = None


def _compta_totes_les_taules(taules_amb_capcalera, file_gender=None) -> ClassificationCounts:
    """Suma els comptadors de totes les (capçalera, files) d'un PDF,
    calculant `detect_columns` PER CADA TAULA per separat (no una sola
    vegada amb la capçalera de la 1a pàgina) -- vegeu la nota a
    `descarrega_i_parseja_pdf`."""
    total = ClassificationCounts()
    for header_cells, files in taules_amb_capcalera:
        col_map = detect_columns(header_cells)
        c = count_from_rows(files, col_map, file_gender=file_gender)
        total.total += c.total
        total.h += c.h
        total.d += c.d
        total.sexe_desconegut += c.sexe_desconegut
    return total


def _valida_parsing_taules(taules_amb_capcalera, counts) -> Optional[str]:
    """Política d'auditabilitat: un PDF de classificació trobat però amb 0
    classificats gairebé segur vol dir que pdfplumber no ha sabut extreure
    la taula, NO que la cursa tingui 0 participants de veritat. Mai es dona
    per bo en silenci: es marca com a error perquè es revisi a mà."""
    if not taules_amb_capcalera:
        return "no s'ha trobat cap taula al PDF (format no reconegut per pdfplumber)"
    if counts.total == 0:
        return "taula trobada però 0 files vàlides comptades (revisar capçalera/format)"
    return None


def processa_modalitat(session: requests.Session, grup: GrupModalitat) -> ResultatModalitat:
    modalitat_codi = normalize(grup.modalitat_nom)[:30] or "general"

    # --- Cas A: un sol PDF, sense sexe al nom -> sexe per fila (Categoria) ---
    if grup.sense_genere and not grup.masculina and not grup.femenina:
        _text, url = grup.sense_genere
        taules, text_complet = descarrega_i_parseja_pdf(session, url)
        counts = _compta_totes_les_taules(taules)

        error = _valida_parsing_taules(taules, counts)
        if error:
            return ResultatModalitat(
                modalitat_nom=grup.modalitat_nom, modalitat_codi=modalitat_codi,
                estat_esdeveniment="llista_error", error_detall=error,
            )

        stats = extract_header_stats(text_complet)
        return ResultatModalitat(
            modalitat_nom=grup.modalitat_nom, modalitat_codi=modalitat_codi,
            estat_esdeveniment="ok",
            total_classificats=counts.total,
            classificat_h=counts.h, classificat_d=counts.d,
            classificat_total=counts.total,
            classificat_sexe_desconegut=counts.sexe_desconegut,
            inscrits=stats.get("inscrits"),
            DNF_total=stats.get("DNF_total"),
            DSQ_total=stats.get("DSQ_total"),
            DNS_total=stats.get("DNS_total"),
        )

    # --- Cas B: un o dos PDFs amb el sexe al NOM del fitxer -> combinar ---
    if grup.masculina or grup.femenina:
        h = d = desconegut = 0
        errors = []
        for gender_key, entry in (("h", grup.masculina), ("d", grup.femenina)):
            if entry is None:
                continue
            _text, url = entry
            taules, _text_complet = descarrega_i_parseja_pdf(session, url)
            counts = _compta_totes_les_taules(taules, file_gender=gender_key)

            error = _valida_parsing_taules(taules, counts)
            if error:
                errors.append(f"fitxer {gender_key}: {error}")
                continue

            h += counts.h
            d += counts.d
            desconegut += counts.sexe_desconegut

        if errors:
            return ResultatModalitat(
                modalitat_nom=grup.modalitat_nom or "GENERAL", modalitat_codi=modalitat_codi,
                estat_esdeveniment="llista_error", error_detall="; ".join(errors),
            )

        total = h + d + desconegut
        return ResultatModalitat(
            modalitat_nom=grup.modalitat_nom or "GENERAL", modalitat_codi=modalitat_codi,
            estat_esdeveniment="ok",
            total_classificats=total,
            classificat_h=h, classificat_d=d, classificat_total=total,
            classificat_sexe_desconegut=desconegut,
        )

    # No hauria d'arribar mai aquí (agrupa_pdfs_per_modalitat sempre omple
    # com a mínim un dels tres camps) -- es deixa per seguretat.
    return ResultatModalitat(
        modalitat_nom=grup.modalitat_nom, modalitat_codi=modalitat_codi,
        estat_esdeveniment="error", error_detall="grup de PDFs buit inesperat",
    )


# ============================================================
# PROCESSAT COMPLET D'UN ESDEVENIMENT -> llista de files pel csv
# ============================================================

def processa_esdeveniment(session: requests.Session, entrada: EntradaLlistat) -> list:
    """Retorna una llista de dicts (una per modalitat) llestos per escriure
    al csv. Mai llança excepció cap enfora (excepte ErrorTransitori, que
    s'ha de propagar perquè main() ho reintenti a la propera execució)."""
    base = {
        "nom_cursa": entrada.nom_cursa,
        "data": normalitza_data_llistat(entrada.data_text),
        "lloc": None,
        "esport": None,
        "event_id": None,
    }

    try:
        resp = get_amb_reintents(session, entrada.url)
    except ErrorPermanent as e:
        return [{**_fila_buida(), **base, "modalitat_nom": "", "modalitat_codi": "",
                 "estat_esdeveniment": "error", "error_detall": str(e)}]
    except ErrorTransitori:
        raise  # no es marca com fet -> una altra execució de main() ho reintentarà

    fitxa = parseja_fitxa_esdeveniment(resp.text)
    base["lloc"] = fitxa.lloc
    base["esport"] = fitxa.esport
    base["event_id"] = fitxa.event_id
    if fitxa.data:  # la data de la fitxa és més fiable que la del llistat
        base["data"] = fitxa.data

    grups = agrupa_pdfs_per_modalitat(fitxa.pdf_links)
    if not grups:
        return [{**_fila_buida(), **base, "modalitat_nom": "", "modalitat_codi": "",
                 "estat_esdeveniment": "sense_resultats", "error_detall": ""}]

    files = []
    for grup in grups:
        try:
            resultat = processa_modalitat(session, grup)
        except ErrorPermanent as e:
            resultat = ResultatModalitat(
                modalitat_nom=grup.modalitat_nom,
                modalitat_codi=normalize(grup.modalitat_nom) or "general",
                estat_esdeveniment="llista_error", error_detall=str(e),
            )
        fila = {**base, **_dataclass_a_dict(resultat)}
        files.append(fila)
    return files


def _fila_buida() -> dict:
    return {c: None for c in COLUMNES_IMPRESCINDIBLES + COLUMNES_EXTRA}


def _dataclass_a_dict(r: ResultatModalitat) -> dict:
    return {
        "modalitat_nom": r.modalitat_nom, "modalitat_codi": r.modalitat_codi,
        "estat_esdeveniment": r.estat_esdeveniment, "error_detall": r.error_detall,
        "total_classificats": r.total_classificats,
        "classificat_h": r.classificat_h, "classificat_d": r.classificat_d,
        "classificat_total": r.classificat_total,
        "classificat_sexe_desconegut": r.classificat_sexe_desconegut,
        "inscrits": r.inscrits, "DNF_total": r.DNF_total,
        "DSQ_total": r.DSQ_total, "DNS_total": r.DNS_total,
    }

In [93]:
# ============================================================
# CHECKPOINT / RESUME + ESCRIPTURA CSV PROTEGIDA
# ============================================================

def carrega_checkpoint() -> set:
    if RESUME and os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def desa_checkpoint(fets: set) -> None:
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(sorted(fets), f, ensure_ascii=False, indent=1)


def _write_csv(files: list) -> None:
    """Protecció: si el CSV existent té més files que el que s'està a punt
    d'escriure, no sobreescriure -- desar a .NOMES_LECTURA_revisa.csv."""
    if os.path.exists(CSV_PATH):
        with open(CSV_PATH, encoding="utf-8-sig") as f:
            files_existents = sum(1 for _ in f) - 1  # menys la capçalera
        if files_existents > len(files):
            path_revisa = str(CSV_PATH).replace(".csv", ".NOMES_LECTURA_revisa.csv")
            _escriu_csv_a(path_revisa, files)
            print(
                f"AVÍS: el csv existent tenia {files_existents} files i "
                f"n'anàvem a escriure {len(files)} -- no s'ha sobreescrit "
                f"'{CSV_PATH}'. Revisa '{path_revisa}' manualment."
            )
            return
    _escriu_csv_a(CSV_PATH, files)


def _escriu_csv_a(path: str, files: list) -> None:
    # utf-8-sig (afegeix un BOM) perquè l'Excel de Windows detecti UTF-8
    # automàticament en obrir el fitxer -- sense el BOM, Excel assumeix la
    # codificació ANSI del sistema i mostra "però" com "perÃ²", encara que
    # el contingut del fitxer ja sigui UTF-8 correcte.
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=TOTES_LES_COLUMNES)
        writer.writeheader()
        for fila in files:
            writer.writerow({c: fila.get(c) for c in TOTES_LES_COLUMNES})


def _clau_resume(entrada: EntradaLlistat) -> str:
    """Resume per l'URL de la cursa (la coneixem abans de baixar la fitxa,
    a diferència de l'event_id)."""
    return entrada.url


# ============================================================
# ORQUESTRACIÓ PRINCIPAL
# ============================================================

def main():
    fets = carrega_checkpoint()
    totes_les_files = []
    if RESUME and os.path.exists(CSV_PATH):
        with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
            totes_les_files = list(csv.DictReader(f))

    session = crea_sessio()
    comptador_noves = 0

    try:
        for entrada in crawl_llistat(session, max_pagines=MAX_PAGINES_LLISTAT):
            clau = _clau_resume(entrada)
            if RESUME and clau in fets:
                continue

            try:
                files_cursa = processa_esdeveniment(session, entrada)
            except ErrorTransitori as e:
                print(f"[transitori, es reintentarà una altra vegada] {entrada.url}: {e}")
                continue  # NO s'afegeix a `fets` -> es reintentarà

            totes_les_files.extend(files_cursa)
            fets.add(clau)
            comptador_noves += 1
            print(f"[{comptador_noves}] {entrada.nom_cursa} "
                  f"-> {len(files_cursa)} modalitat(s)")

            if comptador_noves % SAVE_EVERY_N_RACES == 0:
                _write_csv(totes_les_files)
                desa_checkpoint(fets)

            time.sleep(REQUEST_DELAY_SECONDS)
    finally:
        _write_csv(totes_les_files)
        desa_checkpoint(fets)
        print(f"Fet. {comptador_noves} curses noves processades aquesta execució, "
              f"{len(totes_les_files)} files en total a '{CSV_PATH}'.")

In [94]:
os.chdir(Path.home() / "OneDrive - Prenomics" / "Escritorio" / "Altres" / "cronofinisher_data")
MAX_PAGINES_LLISTAT = None # <- primer una prova amb 1 pàgina (15 curses) abans del run complet
main()

[1] IX CARRERA DE ROCHE POR EL SINDROME DE RETT -> 5 modalitat(s)
[2] XV CARRERA POPULAR NOCTURNA MALPARTIDA DE CACERES -> 2 modalitat(s)
[3] VIII MARATON CIUDAD DE JAEN -> 3 modalitat(s)
[4] XXXVI TRAVESÍA INTERNACIONAL A NADO “CIUDAD DE CÁDIZ” -> 2 modalitat(s)
[5] XII CXM MANCHA REAL - SIERRA MAGINA 2026 -> 2 modalitat(s)
[6] I MILLA CIUDAD DE TARIFA -> 1 modalitat(s)
[7] VII JORNADA DEPORTIVA SOLIDARIA -> 1 modalitat(s)
[8] IVAFIT CHALLENGE -> 6 modalitat(s)
[9] IX CXM NOCTURNA MONTES DE CEUTA -> 3 modalitat(s)
[10] XXXII CARRERA URBANA SAN FELIPE NERI -> 4 modalitat(s)
[11] CARRERA POPULAR SAN MARTIN DEL TESORILLO -> 5 modalitat(s)
[12] CAMPEONATO ESPAÑA SELECCIONES AUTONOMICAS - KM VERTICAL - CXM -> 8 modalitat(s)
[13] DUAL BATTLE GUADALQUIVIR -> 4 modalitat(s)
[14] III MARATON - MEDIA MARATON - MARATÓN POR EQUIPOS 2026 . EN PISTA TORREPEROGIL -> 3 modalitat(s)
[15] MEMORIAL ISIDRO PULIDO GRAVEL BY GSPORT - COPA DE ESPAÑA - GRAVEL -> 1 modalitat(s)
[16] XIV SUBIDA AL CASTILLO DE 